# 서비스 아키텍처 실습 — 환경 분리 · 모델 라우팅 · 캐시 · Multi-Agent · 비용 최적화

Dev–Staging–Prod 환경 분리, 모델 계층화, Semantic Cache, Multi-Agent 파이프라인, 비용 산정까지
**Agent 서비스를 프로덕션에 올리기 위한 핵심 패턴**을 직접 구현하고 비교합니다.

| 섹션 | 핵심 질문 | 구현하는 것 |
|------|-----------|-------------|
| 환경 분리 | Dev와 Prod가 왜 달라야 하나? | YAML 기반 설정 로더 |
| 모델 라우터 | 모든 요청에 비싼 모델을 써야 하나? | 복잡도 기반 자동 라우팅 |
| Semantic Cache | 비슷한 질문마다 API를 호출해야 하나? | 임베딩 유사도 캐시 |
| Multi-Agent | 하나의 Agent가 모든 걸 해야 하나? | LangGraph 파이프라인 |
| 비용 산정 | 월 비용이 얼마나 나올까? | 토큰 기반 비용 계산기 |

---
## 1. 환경 설정

In [ ]:
# %pip install langchain langchain-openai langchain-community langgraph langfuse pyyaml numpy tiktoken

from dotenv import load_dotenv
load_dotenv(override=True)

import os, json, yaml, time
import numpy as np
from dataclasses import dataclass, field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

print(f"✅ 환경 변수 로드 완료")
print(f"   OPENAI_API_KEY: {'설정됨' if os.environ.get('OPENAI_API_KEY') else '❌ 미설정'}")
print(f"   LANGFUSE:       {'설정됨' if os.environ.get('LANGFUSE_SECRET_KEY') else '❌ 미설정'}")

In [ ]:
# Langfuse Observability 설정
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"✅ Langfuse tracing ON — {os.environ.get('LANGFUSE_BASE_URL', '')}")
else:
    print("⚠️  Langfuse 미설정 — 트레이싱 없이 진행합니다")

lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

---
## 2. Dev–Staging–Prod 환경 분리

프로덕션 장애의 상당수는 **"개발 환경과 운영 환경의 차이"** 에서 발생합니다.

```
❌ 개발자 PC에서 잘 되던 게 운영에서 안 됨
❌ 긴급 패치를 운영에 바로 배포 → 장애
❌ 개발 중 실제 사용자 데이터 사용 → 개인정보 유출
```

환경을 분리하면 이런 문제를 구조적으로 방지할 수 있습니다.

| 환경 | 목적 | 모델 | 데이터 |
|------|------|------|--------|
| **Dev** | 개발·실험 | 저비용 (gpt-5.4-mini) | 합성/샘플 |
| **Staging** | 검증·QA | 운영과 동일 | 익명화 복사본 |
| **Prod** | 실제 서비스 | 운영 모델 (gpt-5.4) | 실제 데이터 |

### 2.1 환경별 설정 파일 생성 (YAML)

코드에 `if env == "prod"` 같은 분기를 넣는 대신, **설정 파일**로 환경을 분리합니다.  
코드는 하나, 설정만 환경별로 다릅니다.

In [ ]:
# 환경별 YAML 설정 파일 생성
configs = {
    "dev": {
        "llm": {"model": "gpt-5.4-mini", "temperature": 0.7, "max_tokens": 1024},
        "cache": {"enabled": False},
        "guardrail": {"enabled": False, "pii_masking": False},
        "logging": {"level": "DEBUG", "sample_rate": 1.0},
    },
    "staging": {
        "llm": {"model": "gpt-5.4", "temperature": 0.0, "max_tokens": 2048},
        "cache": {"enabled": True, "ttl_seconds": 300},
        "guardrail": {"enabled": True, "pii_masking": False},
        "logging": {"level": "INFO", "sample_rate": 1.0},
    },
    "prod": {
        "llm": {"model": "gpt-5.4", "temperature": 0.0, "max_tokens": 2048},
        "cache": {"enabled": True, "ttl_seconds": 3600},
        "guardrail": {"enabled": True, "pii_masking": True},
        "logging": {"level": "WARNING", "sample_rate": 0.1},  # 정상 트래픽 10%만 로깅
    },
}

os.makedirs("config", exist_ok=True)
for env_name, cfg in configs.items():
    with open(f"config/{env_name}.yaml", "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
    print(f"✅ config/{env_name}.yaml 생성")

# Prod 설정 확인
print("\n--- config/prod.yaml ---")
with open("config/prod.yaml") as f:
    print(f.read())

### 2.2 설정 로더 (AppConfig)

YAML 파일을 읽어 하나의 설정 객체로 만듭니다.  
환경 변수 `APP_ENV`로 어떤 설정을 쓸지 결정합니다 — 코드를 바꾸지 않고 환경만 전환됩니다.

In [ ]:
@dataclass
class AppConfig:
    """환경별 설정을 하나의 객체로 관리"""
    env: str
    llm_model: str
    llm_temperature: float
    llm_max_tokens: int
    cache_enabled: bool
    cache_ttl: int
    guardrail_enabled: bool
    pii_masking: bool
    log_level: str
    log_sample_rate: float

    @classmethod
    def from_yaml(cls, env: str) -> "AppConfig":
        """YAML 파일에서 설정 로드"""
        with open(f"config/{env}.yaml") as f:
            cfg = yaml.safe_load(f)
        return cls(
            env=env,
            llm_model=cfg["llm"]["model"],
            llm_temperature=cfg["llm"]["temperature"],
            llm_max_tokens=cfg["llm"]["max_tokens"],
            cache_enabled=cfg.get("cache", {}).get("enabled", False),
            cache_ttl=cfg.get("cache", {}).get("ttl_seconds", 0),
            guardrail_enabled=cfg.get("guardrail", {}).get("enabled", False),
            pii_masking=cfg.get("guardrail", {}).get("pii_masking", False),
            log_level=cfg.get("logging", {}).get("level", "INFO"),
            log_sample_rate=cfg.get("logging", {}).get("sample_rate", 1.0),
        )

    def create_llm(self) -> ChatOpenAI:
        """설정에 맞는 LLM 인스턴스 생성"""
        return ChatOpenAI(
            model=self.llm_model,
            temperature=self.llm_temperature,
            max_tokens=self.llm_max_tokens,
        )


def load_config() -> AppConfig:
    """APP_ENV 환경 변수로 설정 로드 (기본: dev)"""
    env = os.environ.get("APP_ENV", "dev")
    return AppConfig.from_yaml(env)


print("✅ AppConfig 정의 완료")

In [ ]:
# 세 환경의 설정 비교
print(f"{'항목':<20} {'Dev':<20} {'Staging':<20} {'Prod':<20}")
print("=" * 80)

dev_cfg = AppConfig.from_yaml("dev")
stg_cfg = AppConfig.from_yaml("staging")
prd_cfg = AppConfig.from_yaml("prod")

rows = [
    ("LLM 모델",       dev_cfg.llm_model,       stg_cfg.llm_model,       prd_cfg.llm_model),
    ("Temperature",     dev_cfg.llm_temperature,  stg_cfg.llm_temperature,  prd_cfg.llm_temperature),
    ("Max Tokens",      dev_cfg.llm_max_tokens,   stg_cfg.llm_max_tokens,   prd_cfg.llm_max_tokens),
    ("캐시",            dev_cfg.cache_enabled,     stg_cfg.cache_enabled,     prd_cfg.cache_enabled),
    ("PII 마스킹",     dev_cfg.pii_masking,       stg_cfg.pii_masking,       prd_cfg.pii_masking),
    ("로그 레벨",       dev_cfg.log_level,         stg_cfg.log_level,         prd_cfg.log_level),
    ("로그 샘플링",     dev_cfg.log_sample_rate,   stg_cfg.log_sample_rate,   prd_cfg.log_sample_rate),
]

for name, d, s, p in rows:
    print(f"{name:<20} {str(d):<20} {str(s):<20} {str(p):<20}")

### 2.3 환경 전환 시 LLM 동작 비교

같은 코드에 환경만 바꾸면 **모델, temperature, 가드레일**이 모두 달라집니다.  
Dev에서는 자유롭게 실험하고, Prod에서는 일관된 응답을 보장합니다.

In [ ]:
# 같은 질문, 다른 환경 → 다른 동작
question = "Python의 GIL이 무엇인가요?"

for env_name in ["dev", "prod"]:
    cfg = AppConfig.from_yaml(env_name)
    llm = cfg.create_llm()

    print(f"\n{'='*60}")
    print(f"🔧 환경: {env_name.upper()} | 모델: {cfg.llm_model} | temp: {cfg.llm_temperature}")
    print(f"{'='*60}")

    response = llm.invoke(
        [HumanMessage(content=f"{question} 한 문장으로 답하세요.")],
        config=lf_config,
    )
    print(f"응답: {response.content}")
    print(f"토큰: 입력={response.usage_metadata['input_tokens']}, 출력={response.usage_metadata['output_tokens']}")

---
## 3. 모델 계층화 라우터 (Model Router)

모든 요청에 가장 비싼 모델을 쓰면 비용이 폭발합니다.

```
"오늘 날씨 어때?" → gpt-5.4 ($$$) ← 과잉!
"이 논문의 핵심 논지를 3가지 관점에서 비교 분석해줘" → gpt-5.4 ($$$) ← 적절
```

**전략**: 질문의 복잡도를 먼저 분류하고, 단순한 건 저비용 모델, 복잡한 건 고비용 모델로 보냅니다.

| 복잡도 | 모델 | 예시 |
|--------|------|------|
| simple | gpt-5.4-mini | 단답형, 사실 확인, 분류 |
| complex | gpt-5.4 | 분석, 비교, 추론, 창작 |

In [ ]:
# 가격 정보 ($/1M 토큰 기준, 2026년 3월)
PRICING = {
    "gpt-5.4-mini": {"input": 0.40, "output": 1.60},
    "gpt-5.4":      {"input": 2.50, "output": 10.00},
}

def calc_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """토큰 수로 비용 계산 (단위: $)"""
    p = PRICING[model]
    return (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000


class ModelRouter:
    """복잡도 기반 모델 라우터 — 단순 질문은 mini, 복잡한 질문은 full 모델"""

    def __init__(self):
        self.classifier = ChatOpenAI(model="gpt-5.4-mini", temperature=0, max_tokens=10)
        self.models = {
            "simple":  ChatOpenAI(model="gpt-5.4-mini", temperature=0),
            "complex": ChatOpenAI(model="gpt-5.4", temperature=0),
        }
        self.log = []

    def classify(self, query: str) -> str:
        """mini 모델로 복잡도 분류 (비용 거의 0)"""
        response = self.classifier.invoke([
            SystemMessage(content=(
                "Classify query complexity. Reply ONLY 'simple' or 'complex'.\n"
                "- simple: factual, yes/no, lookup, short answer, greeting\n"
                "- complex: multi-step reasoning, analysis, comparison, creative writing"
            )),
            HumanMessage(content=query),
        ], config=lf_config)
        level = response.content.strip().lower()
        return level if level in ("simple", "complex") else "complex"

    def route(self, query: str) -> dict:
        """복잡도 분류 → 적절한 모델로 라우팅 → 비용 기록"""
        level = self.classify(query)
        model_name = "gpt-5.4-mini" if level == "simple" else "gpt-5.4"
        model = self.models[level]

        response = model.invoke(
            [HumanMessage(content=query)],
            config=lf_config,
        )

        usage = response.usage_metadata
        cost = calc_cost(model_name, usage["input_tokens"], usage["output_tokens"])
        result = {
            "query": query,
            "level": level,
            "model": model_name,
            "response": response.content[:100] + ("..." if len(response.content) > 100 else ""),
            "input_tokens": usage["input_tokens"],
            "output_tokens": usage["output_tokens"],
            "cost": cost,
        }
        self.log.append(result)
        return result


router = ModelRouter()
print("✅ ModelRouter 준비 완료")

In [ ]:
# 다양한 질문으로 라우팅 테스트
test_queries = [
    "Python은 어떤 언어인가요?",                           # simple
    "1 + 1은?",                                             # simple
    "마이크로서비스와 모놀리식 아키텍처를 확장성, 운영 복잡도, 팀 구조 관점에서 비교 분석해주세요.",  # complex
    "Docker란 무엇인가요?",                                 # simple
    "대규모 트래픽 환경에서 Agent 서비스의 Stateless 설계와 세션 관리 전략을 설계해주세요.",         # complex
]

print(f"{'질문':<35} {'분류':<10} {'모델':<15} {'비용($)':<10}")
print("=" * 75)

for q in test_queries:
    result = router.route(q)
    print(f"{q[:33]:<35} {result['level']:<10} {result['model']:<15} ${result['cost']:.6f}")

In [ ]:
# 라우터 비용 vs 전부 gpt-5.4를 쓴 비용 비교
routed_total = sum(r["cost"] for r in router.log)

# 모든 요청을 gpt-5.4로 처리했다면?
always_expensive = sum(
    calc_cost("gpt-5.4", r["input_tokens"], r["output_tokens"])
    for r in router.log
)

savings_pct = (1 - routed_total / always_expensive) * 100 if always_expensive > 0 else 0

print(f"📊 비용 비교 ({len(router.log)}건 요청)")
print(f"   전부 gpt-5.4 사용:  ${always_expensive:.6f}")
print(f"   라우터 사용:         ${routed_total:.6f}")
print(f"   절감율:              {savings_pct:.1f}%")
print(f"\n💡 단순 질문이 많을수록 절감 효과가 커집니다.")

---
## 4. Semantic Cache

동일하거나 유사한 질문이 반복될 때, 매번 LLM API를 호출하면 **비용 낭비**입니다.

```
"Python이 뭐야?"       → LLM 호출 (캐시 미스)
"Python이 무엇인가요?" → 캐시 히트! (유사도 0.95) → 저장된 응답 반환
"파이썬은 뭔가요?"     → 캐시 히트! (유사도 0.93) → 저장된 응답 반환
```

**Exact Cache**는 문자열이 정확히 일치해야 히트됩니다.  
**Semantic Cache**는 임베딩 유사도로 판단하므로 표현이 달라도 같은 의미면 히트됩니다.

In [ ]:
class SemanticCache:
    """임베딩 유사도 기반 Semantic Cache"""

    def __init__(self, similarity_threshold: float = 0.92):
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        self.threshold = similarity_threshold
        self._cache: list[tuple[list[float], str, str]] = []  # (embedding, query, response)
        self.stats = {"hits": 0, "misses": 0}

    def _cosine_similarity(self, a, b) -> float:
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    def get(self, query: str) -> dict | None:
        """캐시에서 유사 질문 검색. 히트 시 저장된 응답 반환."""
        if not self._cache:
            self.stats["misses"] += 1
            return None

        query_emb = self.embeddings.embed_query(query)
        best_sim, best_match = 0.0, None

        for emb, cached_q, cached_resp in self._cache:
            sim = self._cosine_similarity(query_emb, emb)
            if sim > best_sim:
                best_sim, best_match = sim, (cached_q, cached_resp)

        if best_sim >= self.threshold and best_match:
            self.stats["hits"] += 1
            return {
                "cached_query": best_match[0],
                "response": best_match[1],
                "similarity": round(best_sim, 4),
            }

        self.stats["misses"] += 1
        return None

    def set(self, query: str, response: str):
        """캐시에 질문-응답 쌍 저장"""
        emb = self.embeddings.embed_query(query)
        self._cache.append((emb, query, response))

    @property
    def hit_rate(self) -> float:
        total = self.stats["hits"] + self.stats["misses"]
        return self.stats["hits"] / total if total > 0 else 0.0


cache = SemanticCache(similarity_threshold=0.90)
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
print("✅ SemanticCache 준비 완료 (threshold=0.90)")

In [ ]:
# 캐시 실험: 같은 의미의 질문을 다양한 표현으로 던져보기
queries = [
    "Python이 뭐야?",                    # 1) 캐시 미스 → LLM 호출
    "Python이 무엇인가요?",               # 2) 캐시 히트? (같은 의미, 다른 표현)
    "파이썬은 어떤 프로그래밍 언어야?",     # 3) 캐시 히트? (한글 표현)
    "JavaScript가 뭐야?",                 # 4) 캐시 미스 (다른 주제)
    "자바스크립트란 무엇인가요?",           # 5) 캐시 히트?
]

for i, q in enumerate(queries, 1):
    cached = cache.get(q)
    if cached:
        print(f"[{i}] ✅ 캐시 히트! (유사도: {cached['similarity']:.4f})")
        print(f"    질문:     {q}")
        print(f"    캐시원본: {cached['cached_query']}")
        print(f"    응답:     {cached['response'][:80]}...")
    else:
        # 캐시 미스 → LLM 호출
        response = llm.invoke(
            [HumanMessage(content=f"{q} 두 문장으로 답하세요.")],
            config=lf_config,
        )
        cache.set(q, response.content)
        print(f"[{i}] ❌ 캐시 미스 → LLM 호출")
        print(f"    질문: {q}")
        print(f"    응답: {response.content[:80]}...")
    print()

In [ ]:
# 캐시 성능 통계
print(f"📊 캐시 통계")
print(f"   히트:   {cache.stats['hits']}건")
print(f"   미스:   {cache.stats['misses']}건")
print(f"   히트율: {cache.hit_rate:.1%}")
print(f"   캐시 항목 수: {len(cache._cache)}개")
print(f"\n💡 실서비스에서 FAQ 패턴이 많을수록 히트율이 높아집니다.")
print(f"   히트율 60%면 LLM 호출 비용이 60% 절감됩니다.")

### 캐시 주의사항

| 문제 | 해결 |
|------|------|
| 문서가 업데이트됐는데 캐시가 구버전 응답 반환 | TTL 설정 + 문서 변경 시 캐시 무효화 |
| threshold가 너무 낮으면 엉뚱한 응답 반환 | 0.90~0.95 사이가 적절 (도메인별 튜닝) |
| 캐시 메모리 무한 증가 | LRU 정책 또는 Redis 같은 외부 저장소 사용 |

---
## 5. Multi-Agent 파이프라인 (LangGraph)

하나의 Agent가 모든 일을 하면 **컨텍스트가 빠르게 차오르고, 디버깅이 어렵고, 한 부분의 실패가 전체를 멈춥니다.**

Multi-Agent 패턴으로 책임을 분리합니다:

```
패턴 1: Orchestrator → Worker (복잡한 장문 Task)
패턴 2: Pipeline 순차 처리 (명확한 처리 흐름)
패턴 3: Parallel Fan-out (다각도 분석)
```

여기서는 **Pipeline 패턴**을 구현합니다: `Researcher → Analyst → Writer`

| Agent | 역할 | 모델 |
|-------|------|------|
| Researcher | 주제 조사, 핵심 사실 수집 | gpt-5.4-mini (저비용) |
| Analyst | 리서치 결과 분석, 인사이트 도출 | gpt-5.4 (고비용, 추론) |
| Writer | 분석 결과를 보고서로 정리 | gpt-5.4-mini (저비용) |

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator


class PipelineState(TypedDict):
    """파이프라인 전체가 공유하는 상태"""
    topic: str                                       # 입력 주제
    research: str                                    # Researcher 출력
    analysis: str                                    # Analyst 출력
    report: str                                      # Writer 출력
    token_usage: Annotated[list[dict], operator.add]  # 각 단계 토큰 사용량


def researcher(state: PipelineState) -> dict:
    """Researcher: 주제에 대한 핵심 사실 3가지 조사"""
    llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
    response = llm.invoke([
        SystemMessage(content="당신은 리서처입니다. 주어진 주제에 대해 핵심 사실 3가지를 조사하세요. 각 사실에 번호를 매기세요."),
        HumanMessage(content=f"주제: {state['topic']}"),
    ], config=lf_config)
    usage = response.usage_metadata
    return {
        "research": response.content,
        "token_usage": [{"agent": "researcher", "model": "gpt-5.4-mini",
                         "input": usage["input_tokens"], "output": usage["output_tokens"]}],
    }


def analyst(state: PipelineState) -> dict:
    """Analyst: 리서치 결과에서 인사이트 도출"""
    llm = ChatOpenAI(model="gpt-5.4", temperature=0)
    response = llm.invoke([
        SystemMessage(content="당신은 분석가입니다. 리서치 결과를 바탕으로 핵심 인사이트 2가지를 도출하세요. 근거를 함께 제시하세요."),
        HumanMessage(content=f"리서치 결과:\n{state['research']}"),
    ], config=lf_config)
    usage = response.usage_metadata
    return {
        "analysis": response.content,
        "token_usage": [{"agent": "analyst", "model": "gpt-5.4",
                         "input": usage["input_tokens"], "output": usage["output_tokens"]}],
    }


def writer(state: PipelineState) -> dict:
    """Writer: 분석 결과를 읽기 쉬운 보고서로 정리"""
    llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0.3)
    response = llm.invoke([
        SystemMessage(content="당신은 보고서 작성자입니다. 분석 내용을 5줄 이내의 간결한 요약 보고서로 정리하세요."),
        HumanMessage(content=f"분석:\n{state['analysis']}"),
    ], config=lf_config)
    usage = response.usage_metadata
    return {
        "report": response.content,
        "token_usage": [{"agent": "writer", "model": "gpt-5.4-mini",
                         "input": usage["input_tokens"], "output": usage["output_tokens"]}],
    }


# 파이프라인 그래프 구성: Researcher → Analyst → Writer
graph = StateGraph(PipelineState)
graph.add_node("researcher", researcher)
graph.add_node("analyst", analyst)
graph.add_node("writer", writer)

graph.add_edge(START, "researcher")
graph.add_edge("researcher", "analyst")
graph.add_edge("analyst", "writer")
graph.add_edge("writer", END)

pipeline = graph.compile()
print("✅ Multi-Agent Pipeline 구성 완료: Researcher → Analyst → Writer")

In [ ]:
# 파이프라인 실행
result = pipeline.invoke(
    {"topic": "AI Agent 서비스의 프로덕션 배포 전략", "token_usage": []},
    config=lf_config,
)

# 각 단계 출력 확인
stages = [
    ("🔍 Researcher", result["research"]),
    ("📊 Analyst", result["analysis"]),
    ("📝 Writer (최종 보고서)", result["report"]),
]

for title, content in stages:
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    print(content)

In [ ]:
# 각 Agent별 토큰 사용량과 비용 분석
print(f"📊 Multi-Agent 파이프라인 비용 분석")
print(f"{'Agent':<15} {'모델':<15} {'입력 토큰':<12} {'출력 토큰':<12} {'비용($)':<10}")
print("=" * 65)

total_cost = 0
for usage in result["token_usage"]:
    cost = calc_cost(usage["model"], usage["input"], usage["output"])
    total_cost += cost
    print(f"{usage['agent']:<15} {usage['model']:<15} {usage['input']:<12} {usage['output']:<12} ${cost:.6f}")

print(f"\n{'합계':<44} ${total_cost:.6f}")

# 만약 전부 gpt-5.4로 처리했다면?
total_if_all_expensive = sum(
    calc_cost("gpt-5.4", u["input"], u["output"]) for u in result["token_usage"]
)
savings = (1 - total_cost / total_if_all_expensive) * 100
print(f"\n💡 모델 계층화 효과: 전부 gpt-5.4 대비 {savings:.1f}% 절감")
print(f"   핵심 추론(Analyst)만 고비용 모델, 나머지는 저비용 모델 사용")

### Multi-Agent 설계 시 판단 기준

처음부터 Multi-Agent로 설계하지 마세요. **단일 Agent로 시작해 병목이 생기면 분리**합니다.

| 조건 | 단일 Agent | Multi-Agent |
|------|-----------|-------------|
| Task 복잡도 | 단순 (~3 스텝) | 복잡 (5+ 스텝) |
| 컨텍스트 길이 | 짧음 | 길거나 누적됨 |
| 도메인 전문성 | 범용 | 여러 전문 영역 |
| 오류 격리 필요 | 낮음 | 높음 |

> **"지금 당장 필요한가?"** 를 항상 물어보세요. 5개 Agent로 시작하면 디버깅이 매우 어려워집니다.

---
## 6. 비용 산정 및 모니터링

LLM API 비용은 트래픽에 따라 **급격히** 증가합니다. 반드시 사전에 추정하고 모니터링해야 합니다.

```
월간 비용 = 일일 요청 수 × 평균 토큰 × 단가 × 30일
```

In [ ]:
def estimate_monthly_cost(
    daily_requests: int,
    avg_input_tokens: int,
    avg_output_tokens: int,
    model: str = "gpt-5.4",
    cache_hit_rate: float = 0.0,
    routing_simple_pct: float = 0.0,
) -> dict:
    """월간 비용 추정기 — 캐시와 라우팅 효과까지 반영"""
    p = PRICING[model]
    p_mini = PRICING["gpt-5.4-mini"]

    # 캐시 적용: 히트된 요청은 LLM 호출 불필요
    llm_requests = daily_requests * (1 - cache_hit_rate)

    # 라우팅 적용: simple 비율은 mini 모델 사용
    complex_requests = llm_requests * (1 - routing_simple_pct)
    simple_requests = llm_requests * routing_simple_pct

    daily_cost_complex = (
        complex_requests * avg_input_tokens * p["input"]
        + complex_requests * avg_output_tokens * p["output"]
    ) / 1_000_000

    daily_cost_simple = (
        simple_requests * avg_input_tokens * p_mini["input"]
        + simple_requests * avg_output_tokens * p_mini["output"]
    ) / 1_000_000

    daily_total = daily_cost_complex + daily_cost_simple
    monthly_total = daily_total * 30

    return {
        "daily_requests": daily_requests,
        "llm_calls_per_day": int(llm_requests),
        "daily_cost": round(daily_total, 2),
        "monthly_cost": round(monthly_total, 2),
        "cache_savings_pct": cache_hit_rate * 100,
        "routing_savings_pct": routing_simple_pct * 100,
    }

print("✅ 비용 추정기 준비 완료")

In [ ]:
# 시나리오별 월간 비용 비교
# 가정: 일 50,000 요청, 평균 입력 500 토큰, 출력 300 토큰

scenarios = {
    "① 최적화 없음 (전부 gpt-5.4)": {
        "cache_hit_rate": 0.0, "routing_simple_pct": 0.0,
    },
    "② 모델 라우팅만 (60% simple)": {
        "cache_hit_rate": 0.0, "routing_simple_pct": 0.6,
    },
    "③ 캐시만 (40% 히트율)": {
        "cache_hit_rate": 0.4, "routing_simple_pct": 0.0,
    },
    "④ 라우팅 + 캐시 (둘 다 적용)": {
        "cache_hit_rate": 0.4, "routing_simple_pct": 0.6,
    },
}

print(f"📊 시나리오별 월간 비용 비교 (일 50,000 요청 기준)")
print(f"{'시나리오':<35} {'일 LLM 호출':<15} {'일 비용':<12} {'월 비용':<12}")
print("=" * 75)

for name, params in scenarios.items():
    est = estimate_monthly_cost(
        daily_requests=50_000,
        avg_input_tokens=500,
        avg_output_tokens=300,
        **params,
    )
    print(f"{name:<35} {est['llm_calls_per_day']:<15,} ${est['daily_cost']:<11,.2f} ${est['monthly_cost']:<11,.2f}")

### 6.1 예산 알림 시뮬레이션

일별 비용이 예산 임계값을 넘으면 알림을 보내는 간단한 모니터 패턴입니다.

- **Soft Limit**: 예산 80% 도달 → 경고 알림
- **Hard Limit**: 예산 100% 초과 → 서비스 제한 또는 중단

In [ ]:
class BudgetMonitor:
    """일별 비용 모니터 — Soft/Hard Limit 알림"""

    def __init__(self, daily_budget: float):
        self.daily_budget = daily_budget
        self.daily_spent = 0.0
        self.alerts = []

    def record(self, cost: float) -> str | None:
        """비용 기록 + 임계값 체크"""
        self.daily_spent += cost
        usage_pct = self.daily_spent / self.daily_budget * 100

        if usage_pct >= 100 and "HARD" not in [a["type"] for a in self.alerts]:
            alert = {"type": "HARD", "msg": f"🚨 HARD LIMIT 초과! {usage_pct:.0f}% (${self.daily_spent:.4f}/${self.daily_budget})"}
            self.alerts.append(alert)
            return alert["msg"]
        elif usage_pct >= 80 and "SOFT" not in [a["type"] for a in self.alerts]:
            alert = {"type": "SOFT", "msg": f"⚠️ SOFT LIMIT 경고! {usage_pct:.0f}% (${self.daily_spent:.4f}/${self.daily_budget})"}
            self.alerts.append(alert)
            return alert["msg"]
        return None

    def status(self) -> str:
        pct = self.daily_spent / self.daily_budget * 100
        return f"일 예산: ${self.daily_budget} | 사용: ${self.daily_spent:.4f} ({pct:.1f}%)"


# 시뮬레이션: 일 예산 $0.01로 설정 (테스트용으로 낮게)
monitor = BudgetMonitor(daily_budget=0.01)

# 이번 실습에서 사용한 라우터 비용을 기록
for r in router.log:
    alert = monitor.record(r["cost"])
    if alert:
        print(alert)

# 파이프라인 비용도 기록
for u in result["token_usage"]:
    cost = calc_cost(u["model"], u["input"], u["output"])
    alert = monitor.record(cost)
    if alert:
        print(alert)

print(f"\n{monitor.status()}")
print(f"발생한 알림: {len(monitor.alerts)}건")

### 6.2 Langfuse에서 확인할 포인트

이 노트북의 모든 LLM 호출은 Langfuse로 트레이싱됩니다. Langfuse 대시보드에서 확인해보세요:

| 확인 항목 | Langfuse 위치 | 왜 중요한가 |
|-----------|---------------|-------------|
| **모델별 비용 비교** | Traces → Cost 필터 | mini vs full 모델의 실제 비용 차이 |
| **레이턴시 분포** | Traces → Latency | 모델별 응답 시간 비교 |
| **Multi-Agent 트레이스** | Traces → 파이프라인 trace 상세 | 각 Agent 노드별 소요 시간과 토큰 |
| **일별 비용 추이** | Dashboard → Daily Cost | 비용 급증 패턴 감지 |
| **토큰 사용량 통계** | Dashboard → Token Usage | 입력/출력 토큰 비율 분석 |

In [ ]:
# Langfuse 플러시 — 모든 트레이스가 전송되도록 보장
if langfuse_handler:
    langfuse_handler.flush()
    print("✅ Langfuse 트레이스 전송 완료")
    print(f"   대시보드: {os.environ.get('LANGFUSE_BASE_URL', 'https://cloud.langfuse.com')}")
else:
    print("⚠️  Langfuse 미설정 — 트레이스 없음")

---
## 7. 핵심 정리

| 주제 | 핵심 원칙 | 이 실습에서 구현한 것 |
|------|-----------|----------------------|
| **환경 분리** | 코드는 하나, 설정만 환경별로 다르게 | YAML 기반 AppConfig 로더 |
| **모델 라우터** | 단순 질문은 저비용, 복잡한 질문만 고비용 | 복잡도 분류 → 자동 라우팅 |
| **Semantic Cache** | 비슷한 질문은 캐시에서 즉시 반환 | 임베딩 유사도 캐시 (threshold 0.90) |
| **Multi-Agent** | 책임 분리, 모델 계층화, 오류 격리 | LangGraph Pipeline (3 Agent) |
| **비용 관리** | 사전 추정 + 실시간 모니터링 + 예산 알림 | 비용 계산기 + BudgetMonitor |

### Scaling 체크리스트

```
□ Agent 서버는 Stateless인가? (세션 상태를 Redis/DB에 저장)
□ 환경별 설정이 YAML/환경변수로 분리되어 있는가?
□ 모델 계층화가 적용되어 있는가? (모든 요청에 비싼 모델 ❌)
□ Semantic Cache로 반복 질문 비용을 줄이고 있는가?
□ 일별 비용 모니터링과 알림이 설정되어 있는가?
□ 카나리 배포로 점진적으로 트래픽을 전환하는가?
```